# Week 2 — Flower Classifier & House Price Predictor

**Theme:** Supervised learning I — k-nearest neighbors & linear regression

Supervised learning means: we have labeled examples (input -> correct answer),
and we want the computer to learn the pattern well enough to predict the answer
for *new*, unseen inputs. There are two flavors:

- **Classification** — the answer is a category (e.g. which species of flower,
  benign vs. malignant)
- **Regression** — the answer is a number (e.g. a price, a score)

**k-Nearest Neighbors (k-NN)** is the simplest way to do either: to predict a
new example, find its `k` closest neighbors among the examples we already
know the answer for, then **vote** (classification) or **average**
(regression). We'll try it on three datasets — two classification, one
regression — then move on to linear regression.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_breast_cancer, fetch_california_housing, load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score

## Part A — k-Nearest Neighbors (Classification & Regression)

세 가지 데이터셋으로 연습합니다: **Iris**(품종 분류), **Breast Cancer**(양성/
악성 분류), **California Housing**(주택 가격 회귀). 먼저 각각 `k=1`로 기본
흐름 — **데이터 나누기 → 모델 만들기(메서드 호출) → 학습 → 평가** — 을
살펴봅니다.

### 1. Iris — 꽃 품종 분류

150개 꽃, 4개 측정값(꽃받침/꽃잎 길이·너비), 3개 품종.

In [ ]:
iris = load_iris()
X_iris, y_iris = iris.data, iris.target
print("Features:", iris.feature_names)
print("Species:", iris.target_names)
print("Shape:", X_iris.shape)

In [ ]:
# 데이터 나누기: 학습용 vs 테스트용 (테스트 데이터는 절대 학습에 쓰지 않습니다)
X_iris_train, X_iris_test, y_iris_train, y_iris_test = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42, stratify=y_iris
)

# 모델 만들기(메서드 호출) -> 학습 -> 평가, 우선 k=1부터 시작해봅니다
knn_iris = KNeighborsClassifier(n_neighbors=1)
knn_iris.fit(X_iris_train, y_iris_train)

predictions = knn_iris.predict(X_iris_test)
accuracy = accuracy_score(y_iris_test, predictions)
print(f"Test accuracy with k=1: {accuracy:.2%}")

**참고:** 아래는 4개 측정값 중 2개(꽃잎 길이/너비)만 써서 k-NN이 실제로
공간을 어떻게 나누는지 시각화한 것입니다 (부드러운 경계를 보여주기 위해
`k=5`로 그렸습니다 — 위 `k=1`보다 경계가 훨씬 덜 들쭉날쭉합니다).

In [ ]:
X_iris2 = X_iris[:, 2:4]  # petal length, petal width
X_iris2_train, X_iris2_test, y_iris2_train, y_iris2_test = train_test_split(
    X_iris2, y_iris, test_size=0.3, random_state=42, stratify=y_iris
)
knn_iris2 = KNeighborsClassifier(n_neighbors=5).fit(X_iris2_train, y_iris2_train)

xx, yy = np.meshgrid(
    np.linspace(X_iris2[:, 0].min() - 0.5, X_iris2[:, 0].max() + 0.5, 200),
    np.linspace(X_iris2[:, 1].min() - 0.5, X_iris2[:, 1].max() + 0.5, 200),
)
Z = knn_iris2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(6, 5))
plt.contourf(xx, yy, Z, alpha=0.25, cmap="viridis")
plt.scatter(X_iris2[:, 0], X_iris2[:, 1], c=y_iris, cmap="viridis", edgecolor="k")
plt.title("k-NN Decision Boundary (petal length vs. width, k=5)")
plt.xlabel(iris.feature_names[2])
plt.ylabel(iris.feature_names[3])
plt.show()

### 2. Breast Cancer — 양성/악성 분류

569개 종양 샘플, 30개 측정값(세포 크기, 질감 등), 2개 클래스
(malignant/benign).

In [ ]:
cancer = load_breast_cancer()
X_cancer, y_cancer = cancer.data, cancer.target
print("Features:", len(cancer.feature_names))
print("Classes:", list(cancer.target_names))
print("Shape:", X_cancer.shape)

In [ ]:
X_cancer_train, X_cancer_test, y_cancer_train, y_cancer_test = train_test_split(
    X_cancer, y_cancer, test_size=0.3, random_state=42, stratify=y_cancer
)

knn_cancer = KNeighborsClassifier(n_neighbors=1)
knn_cancer.fit(X_cancer_train, y_cancer_train)

predictions = knn_cancer.predict(X_cancer_test)
accuracy = accuracy_score(y_cancer_test, predictions)
print(f"Test accuracy with k=1: {accuracy:.2%}")

### 3. California Housing — 주택 가격 회귀

k-NN은 회귀에도 쓸 수 있습니다 — 다수결 투표 대신, 가장 가까운 `k`개
이웃의 **타깃값을 평균**냅니다. 20,640개 캘리포니아 지역구, 8개 특징
(소득, 방 개수, 위치 등)으로 중간 주택 가격(단위: $100,000)을 예측합니다.

*(처음 실행할 때 데이터를 내려받느라 몇 초 걸릴 수 있습니다.)*

In [ ]:
housing = fetch_california_housing()
X_housing, y_housing = housing.data, housing.target
print("Features:", housing.feature_names)
print("Shape:", X_housing.shape)

In [ ]:
X_housing_train, X_housing_test, y_housing_train, y_housing_test = train_test_split(
    X_housing, y_housing, test_size=0.3, random_state=42
)

# 모델 만들기 -> 학습 -> 평가. 분류가 아니라 회귀이므로 Classifier가 아닌
# Regressor를 씁니다 -- 나머지 흐름은 완전히 동일합니다.
knn_housing = KNeighborsRegressor(n_neighbors=1)
knn_housing.fit(X_housing_train, y_housing_train)

predictions = knn_housing.predict(X_housing_test)
print(f"R^2 with k=1:  {r2_score(y_housing_test, predictions):.3f}")
print(f"RMSE with k=1: {mean_squared_error(y_housing_test, predictions) ** 0.5:.3f}  (unit: $100,000)")

## k 값을 바꿔보면 어떻게 될까?

지금까지는 세 데이터셋 모두 `k=1`만 써봤습니다 — 가장 가까운 이웃 **딱
하나**에만 의존하는 거라 노이즈에 민감할 수 있습니다. `k`를 늘리면 성능이
어떻게 달라질까요?

아래 세 칸에서 직접 실험해보세요 — 각 데이터셋에 대해 `k`를 1부터 20까지
바꿔가며 성능(분류는 accuracy, 회귀는 RMSE)을 계산해서 리스트에 저장하고,
`k`에 따라 어떻게 변하는지 그래프로 그려보세요. (힌트: 위에서 `k=1`로 했던
코드를 그대로 가져와서 `for k in range(1, 21):` 반복문 안에 넣고, 매번 새
모델을 만들어 학습·평가한 뒤 결과를 리스트에 추가하면 됩니다.)

In [ ]:
# TODO: Iris에서 k=1~20까지 정확도를 계산해 리스트에 저장하고,
# k에 따른 정확도 변화를 그래프로 그려보세요.


In [ ]:
# TODO: Breast Cancer에서 k=1~20까지 정확도를 계산해 리스트에 저장하고,
# k에 따른 정확도 변화를 그래프로 그려보세요.


In [ ]:
# TODO: California Housing에서 k=1~20까지 RMSE를 계산해 리스트에 저장하고,
# k에 따른 RMSE 변화를 그래프로 그려보세요.


## Part B — Regression: Linear Regression

**Idea:** fit a straight line (or plane, in higher dimensions) through the data
that best predicts a numeric target.

The **diabetes** dataset: 442 patients, several health measurements, and a
target that measures disease progression one year later. We'll start with just
one feature — BMI — so we can plot the fitted line directly.

In [ ]:
diabetes = load_diabetes()
bmi = diabetes.data[:, diabetes.feature_names.index("bmi")].reshape(-1, 1)
target = diabetes.target

Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    bmi, target, test_size=0.3, random_state=42
)

reg = LinearRegression()
reg.fit(Xb_train, yb_train)

print(f"Learned line: progression = {reg.coef_[0]:.1f} * bmi + {reg.intercept_:.1f}")

In [ ]:
predictions = reg.predict(Xb_test)
print(f"R^2 score:  {r2_score(yb_test, predictions):.3f}  (1.0 = perfect, 0.0 = no better than guessing the mean)")
print(f"RMSE:       {mean_squared_error(yb_test, predictions) ** 0.5:.1f}")

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(Xb_test, yb_test, alpha=0.6, label="actual")
order = np.argsort(Xb_test[:, 0])
plt.plot(Xb_test[order], predictions[order], color="red", linewidth=2, label="predicted line")
plt.title("Linear Regression: BMI -> Disease Progression")
plt.xlabel("BMI (standardized)")
plt.ylabel("Disease progression score")
plt.legend()
plt.show()

## Try it yourself

1. **Compare k-NN to a "dumb" baseline.** For Breast Cancer, what accuracy
   would you get by always predicting the majority class? Is k-NN actually
   learning something?
2. **Compare to a "dumb" baseline.** What R² would you get if you always
   predicted the *average* disease progression, regardless of BMI? (Hint:
   that's what an R² of 0 means.)
3. **Add a second feature to the regression.** Use both `bmi` and `s5`
   (another column in `diabetes.feature_names`) as inputs to
   `LinearRegression` — does the R² improve? 아래 셀에 두 특징을 합쳐서
   학습/테스트 데이터를 준비하는 코드를 미리 작성해뒀습니다 — 이어서
   `LinearRegression`으로 학습시키고 R²를 계산해서 비교해보세요.

In [ ]:
# bmi와 s5, 두 특징을 하나의 입력 데이터로 합칩니다
s5 = diabetes.data[:, diabetes.feature_names.index("s5")].reshape(-1, 1)
X_two = np.hstack([bmi, s5])  # (442, 1)짜리 두 배열을 옆으로 이어붙여 (442, 2) 데이터를 만듭니다

Xb2_train, Xb2_test, yb2_train, yb2_test = train_test_split(
    X_two, target, test_size=0.3, random_state=42
)

# TODO: 위에서 나눈 Xb2_train/yb2_train으로 LinearRegression을 학습시키고,
# Xb2_test에 대한 예측의 R² 점수를 계산해서 위 bmi 단일 특징 모델과 비교해보세요.


---
## 🎯 캡스톤: 이번 학기 성적 위험도 예측기

가상의 선배 150명의 "주당 공부시간 / 출석률 / 평균 수면시간 -> 기말 점수" 기록을 드립니다. 이 데이터로 **k-NN 분류기**(위험군 Safe/Warning/Danger 예측)와 **선형회귀**(예상 점수 예측)를 직접 만들어보고, 마지막엔 **여러분 자신의 예상 습관**을 입력해서 결과를 확인해보세요.

**확장 아이디어:** 학기 말에 실제 본인의 공부시간/출석/수면 기록과 실제 성적을 몇 학기치 모아서 `students_df`를 바꿔치기하면, 진짜 "내 성적 예측기"가 됩니다.

In [ ]:
# 더미 데이터 생성 (실행만 하면 됩니다)
import pandas as pd
rng = np.random.default_rng(7)
n_students = 150

weekly_study_hours = np.clip(rng.normal(10, 4, n_students), 0, 25)
attendance_rate = np.clip(rng.normal(0.85, 0.12, n_students), 0.4, 1.0)
sleep_hours_avg = np.clip(rng.normal(6.5, 1.2, n_students), 3, 10)

final_score = (
    20
    + 2.2 * weekly_study_hours
    + 45 * attendance_rate
    + 1.5 * sleep_hours_avg
    + rng.normal(0, 6, n_students)
)
final_score = np.clip(final_score, 0, 100)

def to_risk(score):
    if score >= 80:
        return "Safe"
    elif score >= 60:
        return "Warning"
    return "Danger"

students_df = pd.DataFrame({
    "weekly_study_hours": weekly_study_hours.round(1),
    "attendance_rate": attendance_rate.round(2),
    "sleep_hours_avg": sleep_hours_avg.round(1),
    "final_score": final_score.round(1),
})
students_df["risk"] = students_df["final_score"].apply(to_risk)
students_df.head()

### 여러분의 과제

1. `students_df`에서 `weekly_study_hours`, `attendance_rate`, `sleep_hours_avg` 3개를 입력(X)으로, `risk`를 정답(y)으로 하여 **k-NN 분류기**를 학습시키고 테스트 정확도를 출력하세요. (Part A 코드를 참고하세요: `train_test_split`, `KNeighborsClassifier`, `accuracy_score`)
2. 같은 3개 입력으로 `final_score`(숫자)를 예측하는 **선형회귀 모델**을 학습시키고 R² 점수를 출력하세요. (Part B 코드 참고: `LinearRegression`, `r2_score`)
3. 아래에 **여러분 자신의 예상 습관**(예상 주당 공부시간, 예상 출석률, 예상 평균 수면시간)을 숫자로 입력하고, 학습된 두 모델로 (a) 위험군과 (b) 예상 점수를 각각 예측해서 출력해보세요.

In [ ]:
# TODO 1: k-NN 분류기로 risk(Safe/Warning/Danger)를 예측하는 모델을 학습하고 테스트 정확도를 출력하세요.


# TODO 2: 선형회귀로 final_score를 예측하는 모델을 학습하고 R^2를 출력하세요.


# TODO 3: 나의 예상 습관을 입력하고, 위 두 모델로 위험군과 예상 점수를 예측해보세요.
my_weekly_study_hours = None   # 예: 8
my_attendance_rate = None      # 예: 0.9
my_sleep_hours_avg = None      # 예: 6.5